# dist-send-recv-pair — worked example 3: Pairwise exchange: every rank sends to its right neighbor and receives from its left

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

In a pairwise exchange, each rank simultaneously sends to one neighbor and receives from another. With a ring topology, rank `r` sends to `(r+1) % W` and receives from `(r-1) % W`. Care is needed to ensure the overall set of send/recv pairs is matched — every send must have a corresponding receive, and since both happen at roughly the same time, the operations must not deadlock. With `gloo`, calling `send` then `recv` works because `gloo`'s send buffers the outgoing message.

## Worked solution

World size 4. Each rank holds a value: rank 0 → 100, rank 1 → 200, rank 2 → 300, rank 3 → 400.

**Rank 0:** sends `[100]` to rank 1, receives from rank 3 into `recv_buf`. After exchange: holds `[400]`.
**Rank 1:** sends `[200]` to rank 2, receives from rank 0 into `recv_buf`. After exchange: holds `[100]`.
**Rank 2:** sends `[300]` to rank 3, receives `[200]` from rank 1.
**Rank 3:** sends `[400]` to rank 0, receives `[300]` from rank 2.

Each rank ends up holding the value its left neighbor had. The key pattern: compute `right = (rank + 1) % world_size` and `left = (rank - 1) % world_size`, send to right, receive from left.

In [ ]:
import os
import torch
import torch.distributed as dist
import datetime

def pairwise_exchange_worker(rank, world_size, port, my_value, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo', rank=rank, world_size=world_size,
        timeout=datetime.timedelta(seconds=20)
    )
    right = (rank + 1) % world_size
    left  = (rank - 1) % world_size

    send_buf = torch.tensor([float(my_value)], dtype=torch.float32)
    recv_buf = torch.zeros(1, dtype=torch.float32)

    # Send to right, then receive from left.
    # gloo buffers the outgoing send, so this order is safe.
    dist.send(send_buf, dst=right)
    dist.recv(recv_buf, src=left)

    out_queue.put((rank, recv_buf.item()))
    dist.destroy_process_group()

# Illustrate the ring topology for world_size=4
world_size = 4
values = {0: 100.0, 1: 200.0, 2: 300.0, 3: 400.0}
print('Before exchange:', values)
expected = {r: values[(r - 1) % world_size] for r in range(world_size)}
print('Expected after exchange (each rank gets its left neighbor):', expected)